# Task 1. Define a tool-using agent using ADK

In [ ]:
cd ~
git clone --depth 1 --filter=blob:none --sparse https://github.com/CloudVLab/lab-assets.git temp-repo && cd temp-repo && git sparse-checkout set labs/gsp541/multi-agent-system-skills-starter && cd .. && mv temp-repo/labs/gsp541/multi-agent-system-skills-starter . && rm -rf temp-repo
cd multi-agent-system-skills-starter

In [ ]:
uv sync

In [ ]:
export REGION="asia-east1"
export MODEL="gemini-3.5-flash"
cat << EOF > .env
export GOOGLE_CLOUD_PROJECT="qwiklabs-gcp-04-173fb8337d11"
export GOOGLE_CLOUD_LOCATION="global"
export GOOGLE_GENAI_USE_VERTEXAI=true
export REGION=$REGION
export MODEL=$MODEL
EOF
source .env

In [ ]:
gcloud storage cp agents/researcher/skills/SKILL.md gs://qwiklabs-gcp-04-173fb8337d11-agent-bucket/agents/researcher/skills/SKILL.md
gcloud storage cp agents/researcher/agent.py gs://qwiklabs-gcp-04-173fb8337d11-agent-bucket/agents/researcher/agent.py

In [ ]:
export REGION="asia-east1"
export MODEL="gemini-3.5-flash"
export PROJECT_ID=$(gcloud config get-value project)
export BUCKET_NAME="${PROJECT_ID}-agent-bucket"

cd ~
git clone --depth 1 --filter=blob:none --sparse https://github.com/GoogleCloudPlatform/training-data-analyst.git temp-repo
cd temp-repo
git sparse-checkout set self-paced-labs/agents/build-multi-agent-system/multi-agent-system-skills-starter
cd ..
mv temp-repo/self-paced-labs/agents/build-multi-agent-system/multi-agent-system-skills-starter .
rm -fr temp-repo
cd multi-agent-system-skills-starter

cat << EOF > .env
export GOOGLE_CLOUD_PROJECT="${PROJECT_ID}"
export GOOGLE_CLOUD_LOCATION="${REGION}"
export GOOGLE_GENAI_USE_VERTEXAI=true
export REGION="${REGION}"
export MODEL="${MODEL}"
EOF
source .env

In [ ]:
cat << EOF > agents/researcher/skills/SKILL.md
---
name: skills
description: Gathers detailed information on a topic using Google Search
---
You are an expert researcher.
Use Google Search to gather detailed, up-to-date facts on the requested topic.

Do not try to invoke any skills or tools directly. Do not output raw function calls or JSON payloads. Do not call any function named 'skills_search'. Provide your research findings directly as a plain text summary.
EOF

cat << EOF > agents/researcher/agent.py
import os
import pathlib
from google.adk.agents import Agent
from google.adk.skills import load_skill_from_dir
from google.adk.tools.skill_toolset import SkillToolset

MODEL = os.environ.get("MODEL", "gemini-2.5-flash")
skill_dir = pathlib.Path(__file__).parent / "skills"
researcher_skill = load_skill_from_dir(skill_dir)
researcher_toolset = SkillToolset(skills=[researcher_skill])

researcher = Agent(
    name="researcher",
    model=MODEL,
    tools=[researcher_toolset],
)
root_agent = researcher

EOF

gcloud storage cp agents/researcher/skills/SKILL.md gs://${BUCKET_NAME}/agents/researcher/skills/SKILL.md
gcloud storage cp agents/researcher/agent.py gs://${BUCKET_NAME}/agents/researcher/agent.py

# Task 2. Implement structured output validation

In [ ]:
cat << 'EOF' > agents/judge/agent.py
import os
from typing import Literal
from google.adk.agents import Agent
from google.adk.apps.app import App
from pydantic import BaseModel, Field

MODEL = os.environ.get("MODEL", "gemini-2.5-flash")

class JudgeFeedback(BaseModel):
  status: Literal["pass", "fail"] = Field(
      description="Whether the research is sufficient ('pass') or needs more work ('fail')."
  )
  feedback: str = Field(
      description="Detailed feedback on what is missing. If 'pass', a brief confirmation."
  )

judge = Agent(
    name="judge",
    model=MODEL,
    instruction="""
    You are a strict editor.
    Evaluate the 'research_findings' against the user's original request.
    If the findings are missing key info, return status='fail'.
    If they are comprehensive, return status='pass'.
    """,
    output_schema=JudgeFeedback,
    disallow_transfer_to_parent=True,
    disallow_transfer_to_peers=True,
)

root_agent = judge
EOF

gcloud storage cp agents/judge/agent.py gs://${BUCKET_NAME}/agents/judge/agent.py

# Task 3. Define the Content Builder agent

In [ ]:
cat << 'EOF' > agents/content_builder/skills/SKILL.md
---
name: skills
description: Transforms research findings into a formatted course module and verifies structure.
---
You are an expert course creator.
Take the approved research findings and transform them into a well-structured, engaging course module.

**Formatting Rules:**
1. Start with a main title using a single `#` (H1).
2. Use `##` (H2) for main section headings.

Before finalizing your output, always run the validation script tool 'check_format' to verify the formatting of the generated course module.
EOF

cat << 'EOF' > agents/content_builder/skills/scripts/check_format.py
def check_format(text: str) -> dict:
    # Perform basic heading checks
    has_h1 = text.strip().startswith("# ")
    has_h2 = "## " in text
    if has_h1 and has_h2:
      return {"success": True, "message": "Formatting is valid."}
    else:
      missing = []
      if not has_h1:
        missing.append("H1 heading (# Title)")
      if not has_h2:
        missing.append("H2 section heading (## Section)")
      return {"success": False, "message": f"Missing: {', '.join(missing)}"}
EOF

cat << 'EOF' > agents/content_builder/agent.py
import os
import pathlib
from google.adk.agents import Agent
from google.adk.skills import load_skill_from_dir
from google.adk.tools.skill_toolset import SkillToolset

MODEL = os.environ.get("MODEL", "gemini-2.5-flash")
skill_dir = pathlib.Path(__file__).parent / "skills"
content_builder_skill = load_skill_from_dir(skill_dir)
content_builder_toolset = SkillToolset(skills=[content_builder_skill])

content_builder = Agent(
    name="content_builder",
    model=MODEL,
    tools=[content_builder_toolset],
)

root_agent = content_builder
EOF

gcloud storage cp agents/content_builder/skills/SKILL.md gs://${BUCKET_NAME}/agents/content_builder/skills/SKILL.md
gcloud storage cp agents/content_builder/skills/scripts/check_format.py gs://${BUCKET_NAME}/agents/content_builder/skills/scripts/check_format.py
gcloud storage cp agents/content_builder/agent.py gs://${BUCKET_NAME}/agents/content_builder/agent.py

# Task 4. Configure remote connections and workflow

In [ ]:
cat << 'EOF' > agents/orchestrator/agent.py
import os
import json
from typing import AsyncGenerator
from google.adk.agents import BaseAgent, LoopAgent, SequentialAgent
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.events import Event, EventActions
from google.adk.agents.invocation_context import InvocationContext
from google.adk.agents.callback_context import CallbackContext

from authenticated_httpx import create_authenticated_client

# --- Callbacks ---
def create_save_output_callback(key: str):
    """Creates a callback to save the agent's final response to session state."""
    def callback(callback_context: CallbackContext, **kwargs) -> None:
        ctx = callback_context
        # Find the last event from this agent that has content
        for event in reversed(ctx.session.events):
            if event.author == ctx.agent_name and event.content and event.content.parts:
                text = event.content.parts[0].text
                if text:
                    # Try to parse as JSON if it looks like it, for judge_feedback
                    if key == "judge_feedback" and text.strip().startswith("{"):
                        try:
                            ctx.state[key] = json.loads(text)
                        except json.JSONDecodeError:
                            ctx.state[key] = text
                    else:
                        ctx.state[key] = text
                    print(f"[{ctx.agent_name}] Saved output to state['{key}']")
                    return
    return callback

# Remote Agent Addresses
RESEARCHER_URL = os.environ.get(
    "RESEARCHER_AGENT_CARD_URL", "http://localhost:8001/a2a/agent/.well-known/agent-card.json"
)
JUDGE_URL = os.environ.get(
    "JUDGE_AGENT_CARD_URL", "http://localhost:8002/a2a/agent/.well-known/agent-card.json"
)
CONTENT_BUILDER_URL = os.environ.get(
    "CONTENT_BUILDER_AGENT_CARD_URL", "http://localhost:8003/a2a/agent/.well-known/agent-card.json"
)

# Connect to Researcher (Port 8001)
researcher = RemoteA2aAgent(
    "researcher",
    agent_card=RESEARCHER_URL,
    httpx_client=create_authenticated_client(RESEARCHER_URL),
    after_agent_callback=create_save_output_callback("research_findings"),
)

# TODO 1: Complete the remote connection to the Judge Agent (Port 8002) below
# Pass create_save_output_callback("judge_feedback") to the after_agent_callback parameter.
judge = RemoteA2aAgent(
    "judge",
    agent_card=JUDGE_URL,
    httpx_client=create_authenticated_client(JUDGE_URL),
    after_agent_callback=create_save_output_callback("judge_feedback")
)

# Connect to Content Builder (Port 8003)
content_builder = RemoteA2aAgent(
    "content_builder",
    agent_card=CONTENT_BUILDER_URL,
    httpx_client=create_authenticated_client(CONTENT_BUILDER_URL),
)

# --- Escalation Checker ---

class EscalationChecker(BaseAgent):
    name: str = "escalation_checker"
    description: str = "Escalates workflow when judge passes findings."

    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        feedback = ctx.session.state.get("judge_feedback")
        print(f"[EscalationChecker] Feedback: {feedback}")

        is_pass = False
        if isinstance(feedback, dict) and feedback.get("status") == "pass":
            is_pass = True
        elif isinstance(feedback, str) and '"status": "pass"' in feedback:
            is_pass = True

        # TODO 2: Complete the escalation logic below.
        if is_pass:
            yield Event(author=self.name, actions=EventActions(escalate=True))
        else:
            yield Event(author=self.name)

escalation_checker = EscalationChecker()

# --- Orchestration ---

# TODO 3: Construct the orchestration loop and sequential pipeline agents below.
research_loop = LoopAgent(
    name="research_loop",
    sub_agents=[researcher, judge, escalation_checker],
    max_iterations=3,
)

root_agent = SequentialAgent(
    name="course_creation_pipeline",
    sub_agents=[research_loop, content_builder],
)

EOF

gcloud storage cp agents/orchestrator/agent.py gs://${BUCKET_NAME}/agents/orchestrator/agent.py

# Task 5. Test the system locally

In [ ]:
cat << 'EOF' > run_local.sh
#!/bin/bash

# Create clean directories for local agent runtime configuration
# This prevents the local agent loader from scanning sibling folders as agents
mkdir -p local_run/researcher local_run/judge local_run/content_builder local_run/orchestrator

# Safe cleanup of ports 8080, 8001-8004
echo "Stopping any existing processes on ports 8080, 8001-8004..."
for port in 8080 8001 8002 8003 8004; do
  pid=$(lsof -t -i:$port 2>/dev/null)
  if [ ! -z "$pid" ]; then
    kill -9 $pid 2>/dev/null
  fi
done

# TODO 1: Set the GOOGLE_CLOUD_PROJECT environment variable using a gcloud command
# Tip: Use $(gcloud config get-value project) to retrieve your active Qwiklabs project
export GOOGLE_CLOUD_PROJECT="$(gcloud config get-value project)"

# TODO 2: Set the GOOGLE_CLOUD_LOCATION environment variable to your assigned regional boundary variable
export GOOGLE_CLOUD_LOCATION="${REGION:-us_central1}"

export GOOGLE_GENAI_USE_VERTEXAI="True"

echo "Starting Researcher Agent on port 8001..."
pushd agents/researcher
uv run adk_app.py --host 0.0.0.0 --port 8001 --a2a ../../local_run/researcher &
RESEARCHER_PID=$!
popd

echo "Starting Judge Agent on port 8002..."
pushd agents/judge
uv run adk_app.py --host 0.0.0.0 --port 8002 --a2a ../../local_run/judge &
JUDGE_PID=$!
popd

echo "Starting Content Builder Agent on port 8003..."
pushd agents/content_builder
uv run adk_app.py --host 0.0.0.0 --port 8003 --a2a ../../local_run/content_builder &
CONTENT_BUILDER_PID=$!
popd

# TODO 3: Configure remote connection URLs for the local agents
# These variables match the environment variables expected by the orchestrator agent.
# Specify the correct localhost endpoints pointing to the ports defined above.
export RESEARCHER_AGENT_CARD_URL=http://localhost:8001/a2a/agent/.well-known/agent-card.json
export JUDGE_AGENT_CARD_URL="http://localhost:8002/a2a/agent/.well-known/agent-card.json"
export CONTENT_BUILDER_AGENT_CARD_URL="http://localhost:8003/a2a/agent/.well-known/agent-card.json"

echo "Starting Orchestrator Agent on port 8004..."
pushd agents/orchestrator
uv run adk_app.py --host 0.0.0.0 --port 8004 ../../local_run/orchestrator &
ORCHESTRATOR_PID=$!
popd

# Wait a bit for them to start up and perform their startup cleanup
sleep 3
# Re-create the empty agent folders to satisfy the ADK AgentLoader check during requests
mkdir -p local_run/researcher/agent local_run/judge/agent local_run/content_builder/agent local_run/orchestrator/agent
sleep 2

echo "Starting Frontend App on port 8080..."
pushd app
export AGENT_SERVER_URL=http://localhost:8004

uv run uvicorn main:app --host 0.0.0.0 --port 8080 &
BACKEND_PID=$!
popd

echo "All agents started!"
echo "Researcher: http://localhost:8001"
echo "Judge: http://localhost:8002"
echo "Content Builder: http://localhost:8003"
echo "Orchestrator: http://localhost:8004"
echo "App Server (Frontend): http://localhost:8080"
echo ""
echo "Press Ctrl+C to stop all agents."

# Wait for all processes
trap "kill $RESEARCHER_PID $JUDGE_PID $CONTENT_BUILDER_PID $ORCHESTRATOR_PID $BACKEND_PID 2>/dev/null; rm -rf local_run; exit" INT
wait
EOF

gcloud storage cp run_local.sh gs://${BUCKET_NAME}/run_local.sh

In [ ]:
cd ~/multi-agent-system-skills-starter
chmod +x run_local.sh
./run_local.sh

# Task 6. Deploy the microservices to Cloud Run

In [ ]:
cd ~/multi-agent-system-skills-starter

In [ ]:
source .env

In [ ]:
: "Deploy Researcher agent service"
gcloud run deploy researcher \
  --source agents/researcher/ \
  --region $REGION \
  --allow-unauthenticated \
  --memory 512Mi \
  --set-env-vars GOOGLE_CLOUD_PROJECT=$GOOGLE_CLOUD_PROJECT,GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION,GOOGLE_GENAI_USE_VERTEXAI="true",MODEL=$MODEL

: "Deploy Judge agent service"
gcloud run deploy judge \
  --source agents/judge/ \
  --region $REGION \
  --allow-unauthenticated \
  --memory 512Mi \
  --set-env-vars GOOGLE_CLOUD_PROJECT=$GOOGLE_CLOUD_PROJECT,GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION,GOOGLE_GENAI_USE_VERTEXAI="true",MODEL=$MODEL

: "Deploy Content Builder agent service"
gcloud run deploy content-builder \
  --source agents/content_builder/ \
  --region $REGION \
  --allow-unauthenticated \
  --memory 512Mi \
  --set-env-vars GOOGLE_CLOUD_PROJECT=$GOOGLE_CLOUD_PROJECT,GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION,GOOGLE_GENAI_USE_VERTEXAI="true",MODEL=$MODEL

In [ ]:
: "Capture service URLs"
RESEARCHER_URL=$(gcloud run services describe researcher --region $REGION --format='value(status.url)')
JUDGE_URL=$(gcloud run services describe judge --region $REGION --format='value(status.url)')
CONTENT_BUILDER_URL=$(gcloud run services describe content-builder --region $REGION --format='value(status.url)')

: "Deploy Orchestrator service linking remote agent cards"
gcloud run deploy orchestrator \
  --source agents/orchestrator/ \
  --region $REGION \
  --allow-unauthenticated \
  --memory 512Mi \
  --set-env-vars RESEARCHER_AGENT_CARD_URL=$RESEARCHER_URL/a2a/agent/.well-known/agent-card.json,JUDGE_AGENT_CARD_URL=$JUDGE_URL/a2a/agent/.well-known/agent-card.json,CONTENT_BUILDER_AGENT_CARD_URL=$CONTENT_BUILDER_URL/a2a/agent/.well-known/agent-card.json,GOOGLE_CLOUD_PROJECT=$GOOGLE_CLOUD_PROJECT,GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION,GOOGLE_GENAI_USE_VERTEXAI="true",MODEL=$MODEL

In [ ]:
: "Capture Orchestrator URL and deploy frontend application"
ORCHESTRATOR_URL=$(gcloud run services describe orchestrator --region $REGION --format='value(status.url)')

gcloud run deploy course-creator \
  --source app \
  --region $REGION \
  --allow-unauthenticated \
  --memory 512Mi \
  --set-env-vars AGENT_SERVER_URL=$ORCHESTRATOR_URL,GOOGLE_CLOUD_PROJECT=$GOOGLE_CLOUD_PROJECT

In [ ]:
gcloud run services describe course-creator --region $REGION --format='value(status.url)'

In [ ]:
gcloud logging read "resource.type=cloud_run_revision AND resource.labels.service_name=orchestrator" --project=$GOOGLE_CLOUD_PROJECT --limit 100 --order=asc --format="value(textPayload)"